In [1]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm
from html_to_markdown import convert, ConversionOptions

# ==============================
# Configuration
# ==============================

# EN: Project name identifier used for naming the final markdown file
# ID: Nama proyek yang digunakan untuk penamaan file markdown akhir
NAME = "alpinejs-3x"

# EN: Base URL of the documentation website
# ID: URL utama dari website dokumentasi
BASE_URL = "https://alpinejs.dev"

# EN: Starting page URL for crawling
# ID: URL halaman awal untuk proses crawling
START_URL = "https://alpinejs.dev/start-here"

# EN: Base directory where all files will be stored
# ID: Direktori dasar tempat semua file akan disimpan
BASE_DIR = "docs/alpinejs-3x"

# EN: Directory to store downloaded HTML files
# ID: Direktori untuk menyimpan file HTML hasil download
OUTPUT_DIR = f"{BASE_DIR}/htmls"

# EN: Final combined markdown output file path
# ID: Path file markdown gabungan akhir
OUTPUT_MD = os.path.join(BASE_DIR, f"{NAME}.md")

# EN: Directory to store individual markdown files
# ID: Direktori untuk menyimpan file markdown satu per satu
OUTPUT_MD_DIR = os.path.join(BASE_DIR, "markdowns")

print("Configuration:")
print(f"BASE_URL   : {BASE_URL}")
print(f"START_URL  : {START_URL}")
print(f"BASE_DIR   : {BASE_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print("-" * 70)

# EN: Create output directory if it does not exist
# ID: Membuat folder output jika belum ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

# EN: Create markdown directory if it does not exist
# ID: Membuat folder markdown jika belum ada
os.makedirs(OUTPUT_MD_DIR, exist_ok=True)

print(f"Output folder is ready: {os.path.abspath(OUTPUT_DIR)}")
print("-" * 70)

print("Markdown")
print(f"Source folder : {os.path.abspath(OUTPUT_DIR)}")
print(f"Output file   : {os.path.abspath(OUTPUT_MD)}")
print(f"Output folder : {os.path.abspath(OUTPUT_MD_DIR)}")
print("-" * 70)

# EN: Custom headers to mimic a real browser request
# ID: Header request agar terlihat seperti browser asli
headers = {
    "User-Agent": "Mozilla/5.0"
}

Configuration:
BASE_URL   : https://alpinejs.dev
START_URL  : https://alpinejs.dev/start-here
BASE_DIR   : docs/alpinejs-3x
OUTPUT_DIR : docs/alpinejs-3x/htmls
----------------------------------------------------------------------
Output folder is ready: D:\Code\crawl\dev-docs-hub\docs\alpinejs-3x\htmls
----------------------------------------------------------------------
Markdown
Source folder : D:\Code\crawl\dev-docs-hub\docs\alpinejs-3x\htmls
Output file   : D:\Code\crawl\dev-docs-hub\docs\alpinejs-3x\alpinejs-3x.md
Output folder : D:\Code\crawl\dev-docs-hub\docs\alpinejs-3x\markdowns
----------------------------------------------------------------------


In [2]:
# ==============================
# Step 1: Fetch start page
# ==============================

print("Fetching start page...")

# EN: Send HTTP GET request to start page
# ID: Mengirim request GET ke halaman awal
response = requests.get(START_URL, headers=headers)
response.raise_for_status()

print("Start page successfully fetched")

# EN: Parse HTML using BeautifulSoup
# ID: Parsing HTML menggunakan BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

Fetching start page...
Start page successfully fetched


In [3]:
# ==============================
# Step 2: Extract all links from the sidebar menu
# ==============================

print("Scanning sidebar menu...")

# EN: Locate the aside element that contains the documentation menu
# ID: Mencari elemen aside yang berisi menu dokumentasi
aside_menu = soup.find("aside")

if aside_menu is None:
    raise Exception("Documentation sidebar not found!")

# EN: Extract all anchor tags with href attribute inside sidebar
# ID: Mengambil semua tag anchor dengan atribut href di dalam sidebar
links = aside_menu.find_all("a", href=True)

seen_urls = set()
final_items = []

for link in links:
    href = link["href"].strip()

    # EN: Only process internal documentation links
    # ID: Hanya memproses link dokumentasi internal
    if not href.startswith("/"):
        continue

    # EN: Ignore anchor links and external links
    # ID: Mengabaikan link anchor dan link eksternal
    if href.startswith("#"):
        continue

    full_url = urljoin(BASE_URL, href)

    # EN: Avoid duplicate URLs
    # ID: Menghindari URL duplikat
    if full_url in seen_urls:
        continue

    seen_urls.add(full_url)

    menu_name = link.get_text(strip=True)

    if not menu_name:
        continue

    # EN: Remove invalid filename characters
    # ID: Menghapus karakter yang tidak valid untuk nama file
    safe_name = re.sub(r'[\\/*?:"<>|]', "", menu_name)

    final_items.append((safe_name, full_url))

print(f"Total menus found: {len(final_items)}")

Scanning sidebar menu...
Total menus found: 53


In [4]:
# ==============================
# Step 3: Crawl each documentation page
# ==============================

print("Starting crawling process...")

for idx, (safe_name, url) in enumerate(tqdm(final_items), start=1):
    try:
        # EN: Send request to documentation page
        # ID: Mengirim request ke halaman dokumentasi
        r = requests.get(url, headers=headers)
        r.raise_for_status()

        # EN: Create 4-digit prefix for ordering
        # ID: Membuat prefix 4 digit untuk urutan file
        prefix = f"{idx:04d}"

        filename = f"{prefix}_{safe_name}.html"
        file_path = os.path.join(OUTPUT_DIR, filename)

        # EN: Save raw HTML to file
        # ID: Menyimpan HTML mentah ke file
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)

    except Exception as e:
        print(f"Failed to download {url}: {e}")

print("Crawling finished.")

Starting crawling process...


100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:28<00:00,  1.88it/s]

Crawling finished.


In [5]:
# ==============================
# Step 4: Convert HTML to individual Markdown files and single Markdown file
# ==============================

# EN: Get all HTML files and sort them
# ID: Mengambil semua file HTML dan mengurutkannya
html_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".html")
])

print(f"Total HTML files found: {len(html_files)}")
print("-" * 70)

# EN: Configure markdown conversion options
# ID: Mengatur opsi konversi markdown
options = ConversionOptions(
    heading_style="atx",
    wrap=False
)

all_markdown_content = []

for filename in tqdm(html_files):
    file_path = os.path.join(OUTPUT_DIR, filename)

    # EN: Read HTML file
    # ID: Membaca file HTML
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, "html.parser")

    # EN: Extract main documentation content div
    # ID: Mengambil div utama yang berisi konten dokumentasi
    content_div = soup.find(
        "div",
        class_="m-auto max-w-3xl px-6 pb-24 text-gray-800 antialiased markdown"
    )

    if content_div is None:
        print(f"Content not found in: {filename}")
        continue

    # =====================================================
    # EN: Remove image-related tags before conversion
    # ID: Menghapus tag terkait gambar sebelum konversi
    # =====================================================

    image_tags = ["img", "svg", "picture", "source"]

    for tag in image_tags:
        found = content_div.find_all(tag)
        if found:
            for element in found:
                element.decompose()

    # EN: Convert cleaned HTML to markdown
    # ID: Mengonversi HTML yang sudah dibersihkan menjadi markdown
    markdown = convert(str(content_div), options)

    md_filename = filename.replace(".html", ".md")
    md_file_path = os.path.join(OUTPUT_MD_DIR, md_filename)

    # EN: Save individual markdown file
    # ID: Menyimpan file markdown satu per satu
    with open(md_file_path, "w", encoding="utf-8") as f:
        f.write(markdown)

    title = md_filename.replace(".md", "")
    section_header = f"\n\n# {title}\n\n"

    all_markdown_content.append(section_header + markdown)

# EN: Combine all markdown sections into one document
# ID: Menggabungkan semua bagian markdown menjadi satu dokumen
final_markdown = "\n\n".join(all_markdown_content)

# EN: Save final combined markdown file
# ID: Menyimpan file markdown gabungan akhir
with open(OUTPUT_MD, "w", encoding="utf-8") as f:
    f.write(final_markdown)

print("-" * 70)
print("Conversion completed.")
print(f"Individual markdown files saved in: {OUTPUT_MD_DIR}")
print(f"Combined markdown file created: {OUTPUT_MD}")

Total HTML files found: 53
----------------------------------------------------------------------


 19%|███████████████▍                                                                  | 10/53 [00:01<00:04,  9.64it/s]

Content not found in: 0008_UI ComponentsDropdownModal.html
Content not found in: 0009_Dropdown.html
Content not found in: 0010_Modal.html


100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:06<00:00,  8.72it/s]

----------------------------------------------------------------------
Conversion completed.
Individual markdown files saved in: docs/alpinejs-3x\markdowns
Combined markdown file created: docs/alpinejs-3x\alpinejs-3x.md
